# Baselines - Link Prediction - `sheaf` lane

One lane of the baseline campaign: every link prediction model that needs
`requirements/baselines-sheaf.txt` -- Householder orthogonal maps used by the two sheaf baselines.

- **csnn** on pubmed, lastfm -- The legacy in-repo sheaf model. Reads the HGB graph through the shared HomogeneousView plus a per-type input projection.
- **nsd** on pubmed, lastfm -- Homogeneous sheaf model, same two adaptations as CSNN. `GeneralSheaf` scales as d^2 per edge -- drop it first if memory bites.

**4 cells.** Lanes are independent, so this notebook installs only its own
group and can run on its own Colab runtime alongside the others. To split this lane
further across machines, keep `NUM_RUNS` and `BASE_SEED` fixed and give each machine a
disjoint `UNIT_START`/`UNIT_COUNT` range; never run the same unit index twice.

Generated by `scripts/generate_baseline_notebooks.py` - edit that, not this file.

## 0. Environment

In [ ]:
from pathlib import Path
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/H2S-NN_test')
else:
    PROJECT_DIR = Path.cwd()

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Working directory:', PROJECT_DIR)

Core first. On a fresh Colab runtime this replaces Colab's own torch, so the cell kills
the runtime afterwards; **rerun from the first cell once it restarts**. Where the pinned
versions are already present it is a no-op and execution continues.

In [ ]:
import importlib.metadata
import os
import signal
import subprocess
import sys

CORE_REQUIREMENTS = PROJECT_DIR / 'requirements' / 'core.txt'
if not CORE_REQUIREMENTS.exists():
    raise FileNotFoundError(f'Missing pinned requirements: {CORE_REQUIREMENTS}')

try:
    installed_torch = importlib.metadata.version('torch')
except importlib.metadata.PackageNotFoundError:
    installed_torch = ''
target_ready = installed_torch.startswith('2.6.') and '+cu124' in installed_torch
print('installed torch distribution:', installed_torch or 'not installed')

core = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--upgrade', '-r', str(CORE_REQUIREMENTS)],
    capture_output=True,
    text=True,
)
if core.returncode != 0:
    print(core.stdout[-3000:])
    print(core.stderr[-3000:])
    raise RuntimeError(f'Core install failed with exit code {core.returncode}.')
print('Core stack installed.')

if not target_ready:
    print('Pinned Torch/CUDA stack installed. Restarting; rerun from the first cell.')
    if IN_COLAB:
        os.kill(os.getpid(), signal.SIGKILL)
    raise RuntimeError('Restart Python, then rerun the notebook from the first cell.')

In [ ]:
# The cell above restarts the runtime after replacing Colab's torch. If that rerun did
# not happen, everything below still imports -- and then fails much later, inside a sweep
# worker, with an error about a symbol rather than about the torch version. This is the
# cheap place to catch it, which is why it runs in every lane.
#
# The version strings alone do not prove much: torch_scatter and torch_sparse are
# compiled against one specific torch ABI, and a mismatched build passes a string
# comparison and then dies on import. So import them. Both are in core.txt, so every
# lane has them, and the core lanes genuinely use them -- H2SGNN's link-prediction path
# and the typed graph-classification path both go through torch_sparse.
import torch
import torch_geometric
import torch_scatter  # noqa: F401  (imported to prove the compiled ABI matches)
import torch_sparse  # noqa: F401

if not torch.__version__.startswith('2.6.'):
    raise RuntimeError(
        f'Expected torch 2.6.*, found {torch.__version__}. The install cell above '
        'restarts the runtime; rerun this notebook from the first cell.'
    )
if torch.version.cuda != '12.4':
    raise RuntimeError(f'Expected the CUDA 12.4 build, found CUDA {torch.version.cuda}.')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__, '| pyg:', torch_geometric.__version__, '| device:', DEVICE)
if DEVICE == 'cuda':
    properties = torch.cuda.get_device_properties(0)
    print('GPU:', properties.name, '|', round(properties.total_memory / 1024**3, 1), 'GiB')
else:
    # Worth stopping for in any lane: this notebook is about to launch hundreds of fits,
    # and a CPU-only Colab runtime turns that into days. Change the runtime type.
    print()
    print('WARNING: no GPU on this runtime. Runtime > Change runtime type > GPU.')

In [ ]:
# One optional group, installed on its own. A failure is reported rather than raised;
# the models it backs are dropped further down, before anything launches.
import subprocess
import sys

from src.baselines.runtime.dependencies import REQUIREMENT_GROUPS, refresh

GROUP = REQUIREMENT_GROUPS['sheaf']


def install_group(group, force: bool = False) -> tuple[bool, str, list[str]]:
    base = [sys.executable, '-m', 'pip', 'install', '--upgrade', '-r', str(group.path)]
    if force:
        base += ['--force-reinstall', '--no-cache-dir']
    result = subprocess.run(base, capture_output=True, text=True)
    refresh()
    if group.is_installed():
        return True, 'installed' + (' (--force-reinstall)' if force else ''), []
    # torch_householder ships only an sdist whose setup.py imports torch at build time;
    # pip's default build isolation hides the installed torch from it.
    retry = subprocess.run(base + ['--no-build-isolation'], capture_output=True, text=True)
    refresh()
    if group.is_installed():
        return True, 'installed (--no-build-isolation)', []
    stderr = retry.stderr or result.stderr
    return False, 'failed', [line for line in stderr.splitlines() if line.strip()][-3:]


if GROUP.is_installed():
    print(f'OK    {GROUP.name} already present and usable')
    GROUP_OK = True
else:
    # `is_installed()` is not just an import check: a group that resolves but is the
    # wrong build counts as absent, which is what forces the reinstall below instead of
    # printing 'already present' over a broken install.
    defect = GROUP.defect() if not GROUP.missing_modules() else None
    if defect:
        print(f'DEFECT {GROUP.name}: {defect}')
        print('       reinstalling from scratch...')
    GROUP_OK, detail, tail = install_group(GROUP, force=bool(defect))
    if GROUP_OK:
        print(f'OK    {GROUP.name} {detail} - {GROUP.summary}')
    else:
        print(f'FAIL  {GROUP.name} - {GROUP.summary}')
        print(f'      missing: {", ".join(GROUP.missing_modules()) or GROUP.defect()}')
        for line in tail:
            print(f'      {line[:160]}')

In [ ]:
# Confirm the lane is usable before spending a session on it.
if not GROUP_OK:
    raise RuntimeError(
        f'{GROUP.name} did not install, and every model in this lane needs it. '
        'Fix the pip failure above; there is nothing here that can run without it.'
    )
print('Lane ready.')

In [ ]:
# The vendored baselines are cloned under vendor/, not pip-installed. Check the
# ones this lane reads before anything trains.
from src.baselines.runtime import available_vendors

REQUIRED_VENDORS = ['csnn', 'nsd']
present = available_vendors()
for name in REQUIRED_VENDORS:
    print(('OK      ' if present.get(name) else 'MISSING '), name)
missing = [name for name in REQUIRED_VENDORS if not present.get(name)]
if missing:
    raise RuntimeError(f'Clone these under vendor/ first: {missing}')

In [ ]:
import wandb

wandb_key = None
if IN_COLAB:
    try:
        from google.colab import userdata
        wandb_key = userdata.get('WANDB_API_KEY')
    except Exception:
        wandb_key = None

if wandb_key:
    wandb.login(key=wandb_key, relogin=True)
else:
    wandb.login()

## 1. Campaign settings

In [ ]:
TASK = 'link_prediction'
LANE = 'sheaf'

# (model, datasets) for this lane only.
CELLS = [
    ('csnn', ['pubmed', 'lastfm']),
    ('nsd' , ['pubmed', 'lastfm']),
]

N_TRIALS = 30
NUM_RUNS = 5
BASE_SEED = 42

# Units are seeds. Give each machine a disjoint range; never run one index twice.
UNIT_START = 0
UNIT_COUNT = None

RESUME = False
OUTPUT_DIR = Path('experiments/baselines')
WANDB_PROJECT = 'h2snn_baseline'
WANDB_MODE = 'online'
WANDB_TAGS = ['baselines', TASK, f'lane-{LANE}']

# Epochs and patience are protocol, not settings: they come from
# src/baselines/budget.py so every model in a table trains under the same budget.
# Passing --epochs here would silently override that for this lane only.
from src.baselines.budget import default_budget

budget = default_budget(TASK)
print(f'{sum(len(d) for _, d in CELLS)} cells x {NUM_RUNS} seeds x {N_TRIALS} trials')
print(f'budget: {budget.epochs} epochs, patience {budget.patience}')

Drop any model whose requirement group did not install. This happens **before** the
launch cells, so a missing library costs those cells and nothing else, and it is
reported rather than silent.

In [ ]:
from src.baselines.runtime.dependencies import models_missing_dependencies

requested = [model for model, _ in CELLS]
blocked = models_missing_dependencies(tuple(requested), TASK)
CELLS = [(model, datasets) for model, datasets in CELLS if model not in blocked]

for name, reason in blocked.items():
    print(f'SKIP  {name}: {reason}')
print(f'{len(CELLS)} of {len(requested)} runnable: ' + ', '.join(m for m, _ in CELLS))
if not CELLS:
    raise RuntimeError('No runnable models in this lane; fix the install failure above.')

## 2. Scope

Every cell must be runnable, and each is labelled with whether its authors published
settings for that exact dataset. Published cells go through `run_baseline_published.py`
in section 3; the rest are tuned in section 4.

In [ ]:
from src.baselines.registry import get_adapter, unsupported_reason

PUBLISHED, TUNED = [], []
for model, datasets in CELLS:
    adapter = get_adapter(model)
    for dataset in datasets:
        reason = unsupported_reason(model, TASK, dataset)
        assert reason is None, f'{model}/{dataset} is not runnable: {reason}'
        published = adapter.published_params(TASK, dataset)
        space = adapter.search_space(TASK, dataset)
        (PUBLISHED if published else TUNED).append((model, dataset))
        status = 'published' if published else 'tune'
        print(f'{model:<12}{dataset:<10}{len(space):>3} params  {status}')
print(f'\n{len(PUBLISHED)} published, {len(TUNED)} to tune')

## 3. Published-hyperparameter runs

Cells whose authors published settings for this exact dataset. No sweep: five seeds at
their values, using **their** epoch budget rather than the campaign's, because
reproducing a reported number means reproducing the budget it was reported at. Writes
`published_summary.json`, so it never overwrites a tuned result.

In [ ]:
import subprocess
import sys

if PUBLISHED:
    by_model = {}
    for model, dataset in PUBLISHED:
        by_model.setdefault(model, []).append(dataset)
    for model, datasets in by_model.items():
        cmd = [
            sys.executable, 'scripts/run_baseline_published.py',
            '--task', TASK,
            '--models', model,
            '--datasets', *datasets,
            '--output_dir', str(OUTPUT_DIR),
            '--num_runs', str(NUM_RUNS),
            '--base_seed', str(BASE_SEED),
        ]
        print(' '.join(cmd))
        subprocess.run(cmd, check=True)
else:
    print('No published cells in this lane.')

## 4. Tuned runs

`--validate_only` first: it resolves every search space and prints the fit count without
running anything.

In [ ]:
def tuning_command(model, datasets, validate=False):
    cmd = [
        sys.executable, 'scripts/run_baseline_tuning.py',
        '--task', TASK,
        '--models', model,
        '--datasets', *datasets,
        '--output_dir', str(OUTPUT_DIR),
        '--n_trials', str(N_TRIALS),
        '--num_runs', str(NUM_RUNS),
        '--base_seed', str(BASE_SEED),
        '--unit_start', str(UNIT_START),
        '--wandb_project', WANDB_PROJECT,
        '--wandb_mode', WANDB_MODE,
        '--wandb_tags', *WANDB_TAGS,
    ]
    if UNIT_COUNT is not None:
        cmd += ['--unit_count', str(UNIT_COUNT)]
    if RESUME:
        cmd.append('--resume')
    if validate:
        cmd.append('--validate_only')
    return cmd


TUNE_BY_MODEL = {}
for model, dataset in TUNED:
    TUNE_BY_MODEL.setdefault(model, []).append(dataset)

for model, datasets in TUNE_BY_MODEL.items():
    subprocess.run(tuning_command(model, datasets, validate=True), check=True)

One launch cell per model, so a failure costs one model rather than the lane. After a
Colab restart set `RESUME = True` in section 1 and re-run the cell: compatible per-seed
states resume their original sweeps, and incompatible ones are preserved while numbered
continuation sweeps are created.

### csnn - PUBMED, LASTFM

The legacy in-repo sheaf model. Reads the HGB graph through the shared HomogeneousView plus a per-type input projection.

In [ ]:
model, datasets = 'csnn', TUNE_BY_MODEL.get('csnn', [])
if datasets:
    cmd = tuning_command(model, datasets)
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('csnn: nothing to tune in this lane (published cells ran in section 3).')

### nsd - PUBMED, LASTFM

Homogeneous sheaf model, same two adaptations as CSNN. `GeneralSheaf` scales as d^2 per edge -- drop it first if memory bites.

In [ ]:
model, datasets = 'nsd', TUNE_BY_MODEL.get('nsd', [])
if datasets:
    cmd = tuning_command(model, datasets)
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('nsd: nothing to tune in this lane (published cells ran in section 3).')

## 5. Aggregate

Collects whatever has finished into one CSV for this lane. Cells that have not run are
absent rather than filled in, so a gap stays visibly a gap.

In [ ]:
import json

import pandas as pd

from src.utils.run_reports import report_metrics_for_task

metrics = report_metrics_for_task(TASK)
wanted = {f'{m}_{TASK}_{d}' for m, ds in CELLS for d in ds}

records = []
for pattern in ('*/tuning_summary.json', '*/published_summary.json'):
    for summary_path in sorted(OUTPUT_DIR.glob(pattern)):
        if summary_path.parent.name not in wanted:
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))['summary']
        model, _, dataset = summary_path.parent.name.partition(f'_{TASK}_')
        row = {
            'model': model,
            'dataset': dataset,
            'source': 'published' if 'published' in summary_path.name else 'tuned',
            'seeds': summary.get('num_units', 0),
        }
        for metric in metrics:
            entry = summary.get('metrics', {}).get(metric, {})
            row[metric] = entry.get('mean')
            row[f'{metric}_std'] = entry.get('std')
        records.append(row)

frame = pd.DataFrame(records)
results_path = Path('results') / f'baselines_{TASK}_{LANE}.csv'
results_path.parent.mkdir(parents=True, exist_ok=True)
frame.to_csv(results_path, index=False)
print(f'{len(frame)} of {len(wanted)} cells finished -> {results_path}')
frame

In [ ]:
# Manuscript form: mean +/- std on the headline metric this task reports.
if not frame.empty:
    headline = metrics[0]
    shown = frame.assign(
        value=frame.apply(
            lambda r: f"{100 * r[headline]:.2f} +/- {100 * r[f'{headline}_std']:.2f}",
            axis=1,
        )
    )
    print(shown.pivot_table(index='model', columns='dataset', values='value',
                            aggfunc='first').to_string())
else:
    print('Nothing finished yet - run section 3 and 4 first.')

## 6. Resume and partition

Per-seed state lives under `OUTPUT_DIR/tuning_state/<cell>/`. Set `RESUME = True` in
section 1 and re-run the relevant launch cell to continue. `pruned` counts trials
Hyperband stopped early, which is a normal outcome and not a failure.

In [ ]:
state_root = OUTPUT_DIR / 'tuning_state'
for state_path in sorted(state_root.glob('*/tuning_state_*.json')):
    if state_path.parent.name not in wanted or 'worker' in state_path.stem:
        continue
    state = json.loads(state_path.read_text(encoding='utf-8'))
    print(
        f"{state_path.parent.name:<40}{state_path.stem:<26}"
        f"done {len(state.get('records', []))}"
        f" attempted {state.get('attempted_trials', 0)}"
        f" pruned {state.get('pruned_trials', 0)}"
        f" failed {state.get('failed_trials', 0)}"
        f"  sweep {state.get('sweep_id')}"
    )